# Churn prediction project

Here x is the feature vector - everything we know about the customer - and y is the target variable. The target is binary, so y belongs to {0, 1}:

1 is the positive class: the customer churned
0 is the negative class: the customer stayed

The output of the model - the score - is the likelihood of churning. In this project, 1 means the customer left the company and 0 means the customer stayed.

We learn g from historical data. We take the customers of the previous month - we already know what happened to them. For each of them we know the demographics, what services they use, what kind of contract they have and how much they pay, and we know whether they stayed or left. The model finds the patterns in this data, and then we apply it to the current customers to score them.

Data preparation

In this unit we download the churn dataset, read it with pandas and prepare it for machine learning: make the column names and values uniform, fix the column that pandas read incorrectly, and translate the target variable to numbers.

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [86]:
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


To see all of them at once we can transpose the dataframe - flip rows and columns - with df.head().T

In [87]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [88]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Making everything uniform

The column names are inconsistent: some start with an uppercase letter (CustomerID, Tenure), some contain spaces (Monthly Charges is MonthlyCharges here, but the values inside columns do contain spaces). The values are also inconsistent - we see No, Yes, No phone service and so on. We lowercase everything and replace spaces with underscores:

In [89]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_variables = list(df.dtypes[df.dtypes == 'str'].index)     # .index returns the column names

for c in categorical_variables:
    df[c] = df[c].str.lower().str.replace(' ', '_')

Fixing the totalcharges column

Let's check the types that pandas inferred: df.dtypes

Two things stand out:

1. seniorcitizen is int64 - it is stored as 0/1 instead of yes/no. That's fine, we will treat it as a categorical variable later.
2. totalcharges is object - a string column - although it clearly contains numbers.

Let's try to convert it to numbers:

In [90]:
pd.to_numeric(df.totalcharges, errors='coerce') # force the conversion with errors='coerce'. Unparseable strings are then replaced with NaN

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

It fails without errors='coerce:

ValueError: Unable to parse string "_" at position 488

The reason: in the original data, missing values in this column were encoded with a space. Our normalization step replaced spaces with underscores, so the missing values became "_", and pandas cannot parse that as a number.

In [91]:
# Save it to tc

tc = pd.to_numeric(df.totalcharges, errors='coerce')

Let's see how many missing values this creates:

In [92]:
tc.isnull().sum() # check how many NaN values we have

np.int64(11)

There are 11 such rows. We can inspect them:

In [93]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


These are customers who just joined - their tenure is small and they haven't been billed yet, so their total charges are not available. For them we simply fill the missing values with zero:

In [94]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

Zero is not always the best replacement, but in this case it is acceptable: it matches the fact that these customers haven't paid anything yet.

Encoding the target variable

The target, churn, still contains strings:

In [95]:
# Checking the target variable

df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: str

For classification we need numbers. We compare the column with 'yes'

this produces a boolean series - and cast it to integers:

In [96]:
df.churn = (df.churn == 'yes').astype(int)

In [97]:
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64